In [43]:
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from sklearn.preprocessing import StandardScaler

In [26]:
games = pd.read_csv("magnus_carlsen_games.csv")

In [27]:
games.sample(5)

,id,player_name,opponent_name,player_rating,opponent_rating,format,date,year,result,player_color,opponent_color,result_raw,moves
1924,1924,Magnus Carlsen,Bigfish1995,3264,3093,Blitz,2023-05-22,2023,Win,black,white,1-0,1. e4 c5 2. Nf3 d6 3. d4 cxd4 4. Nxd4 Nf6 5. N...
262,262,Magnus Carlsen,Gusto_Frutta,2827,2176,Rapid,2018-03-07,2018,Win,black,white,1-0,1. b3 a5 2. a4 e5 3. Bb2 f6 4. e3 d5 5. d3 Nc6...
6036,6036,Magnus Carlsen,theloyalwolf,3360,3030,Blitz,2025-09-21,2025,Win,white,black,1-0,1. d4 Nf6 2. c4 e6 3. Nc3 Bb4 4. a3 Bxc3+ 5. b...
2030,2030,Magnus Carlsen,NikoTheodorou,3285,3123,Blitz,2023-06-08,2023,Draw,white,black,0.5-0.5,1. e4 c5 2. Nc3 Nc6 3. f4 g6 4. Nf3 Bg7 5. a3 ...
4838,4838,Magnus Carlsen,BerserkWerewolf,3317,3181,Blitz,2025-01-15,2025,Win,black,white,1-0,1. d4 Nf6 2. Nf3 d6 3. Nc3 d5 4. Bg5 c6 5. e3 ...


In [28]:
games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6699 entries, 0 to 6698
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               6699 non-null   int64 
 1   player_name      6699 non-null   object
 2   opponent_name    6699 non-null   object
 3   player_rating    6699 non-null   int64 
 4   opponent_rating  6699 non-null   int64 
 5   format           6699 non-null   object
 6   date             6699 non-null   object
 7   year             6699 non-null   int64 
 8   result           6699 non-null   object
 9   player_color     6699 non-null   object
 10  opponent_color   6699 non-null   object
 11  result_raw       6699 non-null   object
 12  moves            6695 non-null   object
dtypes: int64(4), object(9)
memory usage: 680.5+ KB


In [29]:
games.describe()

,id,player_rating,opponent_rating,year
count,6699.000000,6699.000000,6699.000000,6699.000000
mean,3349.000000,3223.602478,2962.436483,2023.460069
std,1933.979059,129.367877,258.192988,1.723239
min,0.000000,2619.000000,259.000000,2014.000000
25%,1674.500000,3212.000000,2907.500000,2023.000000
50%,3349.000000,3258.000000,3026.000000,2024.000000
75%,5023.500000,3299.000000,3095.000000,2025.000000
max,6698.000000,3401.000000,3429.000000,2026.000000


In [30]:
games["result_raw"].value_counts()

result_raw
1-0        4756
0-1        1326
0.5-0.5     617
Name: count, dtype: int64

In [31]:
games["target"] = games["result_raw"].apply(lambda x: 1 if x=="1-0" else 0)

In [32]:
games

,id,player_name,opponent_name,player_rating,opponent_rating,format,date,year,result,player_color,opponent_color,result_raw,moves,target
0,0,Magnus Carlsen,RainnWilson,2862,1200,Rapid,2014-12-14,2014,Win,white,black,1-0,1. e4 g6 2. Nf3 d6 3. d4 Bg7 4. Bc4 Bg4 5. Bxf...,1
1,1,Magnus Carlsen,solskytz,2862,1702,Rapid,2014-12-14,2014,Win,white,black,1-0,1. d4 Nf6 2. c4 e6 3. Nc3 Bb4 4. e3 c5 5. Ne2 ...,1
2,2,Magnus Carlsen,Tildenbeatsu,2862,1200,Rapid,2014-12-14,2014,Win,white,black,1-0,1. e4 e5 2. Nf3 Nc6 3. Bb5 Nf6 4. O-O Nxe4 5. ...,1
3,3,Magnus Carlsen,mtmnfy,2862,1200,Rapid,2014-12-14,2014,Win,white,black,1-0,1. d4 e6 2. e4 d5 3. Nd2 Nc6 4. Ngf3 Nf6 5. e5...,1
4,4,Magnus Carlsen,stepanosinovsky,2862,2360,Rapid,2014-12-14,2014,Loss,white,black,0-1,1. d4 Nf6 2. Bg5 c5 3. d5 Ne4 4. Bc1 e6 5. c4 ...,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6694,6694,Magnus Carlsen,BogdanDeac,2854,2724,Blitz,2026-01-02,2026,Win,white,black,1-0,1. c4 b6 2. Nbc3 g6 3. d3 Bg7 4. b4 c5 5. Bb2 ...,1
6695,6695,Magnus Carlsen,mishanick,2859,2689,Blitz,2026-01-02,2026,Win,black,white,1-0,1. b4 b5 2. Nb3 e6 3. e3 Bf6 4. Bxf6 gxf6 5. B...,1
6696,6696,Magnus Carlsen,jefferyx,2849,2781,Blitz,2026-01-02,2026,Loss,white,black,0-1,1. d4 g6 2. b3 f5 3. g3 Nf7 4. Nd3 Bg7 5. c4 O...,0
6697,6697,Magnus Carlsen,DonkyDonkyDonkey,2853,2645,Blitz,2026-01-02,2026,Win,black,white,1-0,1. f4 g6 2. g3 f5 3. Nc3 Bxc3 4. bxc3 d6 5. Nb...,1


In [33]:
# Automatically find non-numeric columns (except target)
non_numeric_cols = games.select_dtypes(include="object").columns
non_numeric_cols = non_numeric_cols.drop("result_raw")

games = games.drop(columns=non_numeric_cols)

In [34]:
games

,id,player_rating,opponent_rating,year,result_raw,target
0,0,2862,1200,2014,1-0,1
1,1,2862,1702,2014,1-0,1
2,2,2862,1200,2014,1-0,1
3,3,2862,1200,2014,1-0,1
4,4,2862,2360,2014,0-1,0
...,...,...,...,...,...,...
6694,6694,2854,2724,2026,1-0,1
6695,6695,2859,2689,2026,1-0,1
6696,6696,2849,2781,2026,0-1,0
6697,6697,2853,2645,2026,1-0,1


In [35]:
games = pd.get_dummies(games, drop_first=True)
games.isnull().sum()

id                    0
player_rating         0
opponent_rating       0
year                  0
target                0
result_raw_0.5-0.5    0
result_raw_1-0        0
dtype: int64

In [36]:
X = games.drop("target", axis=1)
y = games["target"]

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, 
    random_state = 42,
    stratify = y
)

In [39]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [40]:
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=2000)

In [41]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [44]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0
ROC-AUC  : 1.0


In [46]:
confusion_matrix(y_test, y_pred)

array([[389,   0],
       [  0, 951]])

In [48]:
games.columns

Index(['id', 'player_rating', 'opponent_rating', 'year', 'target',
       'result_raw_0.5-0.5', 'result_raw_1-0'],
      dtype='object')

In [49]:
games = games.drop(columns=["id"])

In [50]:
y.value_counts(normalize=True)

target
1    0.709957
0    0.290043
Name: proportion, dtype: float64